# 01: 気温データの探索的分析 (EDA)

## なぜ気温データがデリバティブの原資産になるのか？

金融デリバティブは通常、**株価**や**金利**のような取引可能な資産を原資産とする。
しかし、エネルギー業界では**気温**が事業収益に直結する：

- **ガス会社**: 冬が暖かいと暖房需要が減り、売上が下がる
- **電力会社**: 夏が涼しいとエアコン需要が減り、売上が下がる
- **農業**: 異常気温は作物の生育に影響

こうした「気温リスク」をヘッジする手段として、1990年代後半にCME（シカゴ・マーカンタイル取引所）で
**気候デリバティブ**の取引が始まった。

### HDD/CDD — 気温リスクの定量化指標

- **HDD (Heating Degree Days)** = max(65°F − T_avg, 0) → 暖房需要の代理変数
- **CDD (Cooling Degree Days)** = max(T_avg − 65°F, 0) → 冷房需要の代理変数
- CME先物は、特定月のHDD/CDD **累積値** に対して取引される（1 HDD/CDD = $20）

このNotebookでは、シカゴの気温データを取得・可視化し、HDD/CDDのパターンを理解する。

In [ ]:
import sys
sys.path.insert(0, "../backend")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import date

from app.services.noaa_client import generate_sample_data
from app.services.degree_days import (
    calculate_daily_degree_days,
    calculate_cumulative,
    calculate_monthly_degree_days,
    DEFAULT_BASE_TEMP_C,
)

# プロットの日本語対応 & スタイル設定
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")

print("Setup complete!")

## 1. 気温データの取得

NOAA APIキーがあれば実データを取得できるが、ここではサンプルデータを使って分析の流れを学ぶ。
サンプルデータはシカゴの気温パターンを模した合成データ（季節成分 + ノイズ）。

In [ ]:
# 3年分のサンプルデータを生成（シカゴの気温パターン）
sample_data = generate_sample_data(
    start_date=date(2021, 1, 1),
    end_date=date(2023, 12, 31),
)

# pandasのDataFrameに変換して分析しやすくする
df = pd.DataFrame([
    {"date": d.date, "tmax": d.tmax, "tmin": d.tmin, "tavg": d.tavg}
    for d in sample_data
])
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")

print(f"データ期間: {df.index.min().date()} ~ {df.index.max().date()}")
print(f"レコード数: {len(df)}")
print(f"\n基本統計量:")
df.describe().round(1)

## 2. 気温の時系列プロット — 季節性の確認

気温データの最も重要な特徴は**強い季節性**。
株価と違って、気温は毎年同じようなパターンを繰り返す。
これが後のモデリング（OU過程）で「平均回帰」として捉えられる。

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# TMAX/TMIN の範囲を塗りつぶし
ax.fill_between(df.index, df["tmin"], df["tmax"], alpha=0.3, color="steelblue", label="TMIN〜TMAX range")
# 平均気温の線
ax.plot(df.index, df["tavg"], color="navy", linewidth=0.8, label="T_avg (daily mean)")
# 基準温度の水平線
ax.axhline(y=DEFAULT_BASE_TEMP_C, color="red", linestyle="--", alpha=0.7, label=f"Base temp = {DEFAULT_BASE_TEMP_C}°C (65°F)")

ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Chicago Daily Temperature — 3 Year Sample Data")
ax.legend(loc="upper right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n基準温度 {DEFAULT_BASE_TEMP_C}°C (赤い破線) より上 → CDD発生（冷房需要）")
print(f"基準温度 {DEFAULT_BASE_TEMP_C}°C より下 → HDD発生（暖房需要）")

## 3. 月別気温分布 — 季節性の統計的確認

箱ひげ図で月ごとの気温分布を確認する。
冬（12-2月）と夏（6-8月）の分布の違いが、HDD/CDDの季節パターンの源泉。

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

df["month"] = df.index.month
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

bp = ax.boxplot(
    [df[df["month"] == m]["tavg"].values for m in range(1, 13)],
    labels=month_labels,
    patch_artist=True,
)

# 寒い月は青、暑い月は赤のグラデーション
colors = plt.cm.coolwarm(np.linspace(0, 1, 12))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(y=DEFAULT_BASE_TEMP_C, color="red", linestyle="--", alpha=0.5, label=f"Base = {DEFAULT_BASE_TEMP_C}°C")
ax.set_ylabel("Average Temperature (°C)")
ax.set_title("Monthly Temperature Distribution — Chicago (3 years)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. HDD/CDD の計算と可視化

HDD/CDDを日次・月次で計算し、その季節パターンを確認する。

**ポイント**: 
- 冬はHDDが大きい（暖房需要）、夏はCDDが大きい（冷房需要）
- CME先物はこの月次累積値に対して取引される

In [ ]:
# HDD/CDD を計算
daily_dd = calculate_daily_degree_days(sample_data)
monthly_dd = calculate_monthly_degree_days(daily_dd)
cum_hdd, cum_cdd = calculate_cumulative(daily_dd)

# DataFrameに変換
df_dd = pd.DataFrame([
    {"date": dd.date, "hdd": dd.hdd, "cdd": dd.cdd, "tavg": dd.tavg}
    for dd in daily_dd
])
df_dd["date"] = pd.to_datetime(df_dd["date"])
df_dd = df_dd.set_index("date")

print(f"3年間の累積HDD: {cum_hdd:.1f}")
print(f"3年間の累積CDD: {cum_cdd:.1f}")
print(f"年平均HDD: {cum_hdd / 3:.1f}  (シカゴ実績の典型値: ~6,000)")
print(f"年平均CDD: {cum_cdd / 3:.1f}  (シカゴ実績の典型値: ~1,000)")

In [ ]:
# 日次HDD/CDD のスタックエリアチャート
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# HDD (上段)
axes[0].fill_between(df_dd.index, df_dd["hdd"], color="steelblue", alpha=0.7)
axes[0].set_ylabel("Daily HDD")
axes[0].set_title("Daily HDD (Heating Degree Days) — Larger in winter")

# CDD (下段)
axes[1].fill_between(df_dd.index, df_dd["cdd"], color="orangered", alpha=0.7)
axes[1].set_ylabel("Daily CDD")
axes[1].set_title("Daily CDD (Cooling Degree Days) — Larger in summer")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. 月次HDD/CDD累積値 — CME先物の原資産

CME気候先物は **月次HDD/CDD累積値** に対して取引される。
例えば「2024年1月シカゴHDD先物」の決済値は、1月全日のHDDの合計。

下のグラフで、各月のHDD（青）とCDD（赤）の累積値を年ごとに比較する。
**年ごとのばらつき** = 気候リスク = デリバティブでヘッジすべきもの。

In [ ]:
# 月次HDD/CDD をDataFrameに変換
df_monthly = pd.DataFrame([
    {"year": m.year, "month": m.month, "hdd": m.hdd, "cdd": m.cdd, "days": m.days_count}
    for m in monthly_dd
])

# 年ごとに月次HDDを並べて比較
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for year in df_monthly["year"].unique():
    yearly = df_monthly[df_monthly["year"] == year]
    axes[0].plot(yearly["month"], yearly["hdd"], "o-", label=str(year), alpha=0.8)
    axes[1].plot(yearly["month"], yearly["cdd"], "o-", label=str(year), alpha=0.8)

axes[0].set_title("Monthly Cumulative HDD by Year")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("HDD")
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_labels)
axes[0].legend()

axes[1].set_title("Monthly Cumulative CDD by Year")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("CDD")
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels)
axes[1].legend()

plt.tight_layout()
plt.show()

print("年ごとのばらつき（特に冬のHDD）が気候リスクの源泉。")
print("このリスクをヘッジするために気候デリバティブが存在する。")

## 6. HDD/CDDの分布 — デリバティブプライシングの基礎

月次HDD/CDDの分布を見ることで、先物・オプションの価格感覚をつかむ。

- **先物価格** ≈ HDD/CDD累積値の期待値（平均）
- **オプション価格** は分布の裾の広がり（分散）に依存する
- 分布が正規分布からずれている場合、単純なBlack-Scholesは使えない → Monte Carloが必要

In [ ]:
# 冬月（1月）のHDDと夏月（7月）のCDDの分布を、日次ベースで確認
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

jan_hdd = df_dd[df_dd.index.month == 1]["hdd"]
jul_cdd = df_dd[df_dd.index.month == 7]["cdd"]

axes[0].hist(jan_hdd, bins=20, color="steelblue", alpha=0.7, edgecolor="white")
axes[0].axvline(jan_hdd.mean(), color="navy", linestyle="--", label=f"Mean = {jan_hdd.mean():.1f}")
axes[0].set_title("January Daily HDD Distribution")
axes[0].set_xlabel("HDD")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].hist(jul_cdd, bins=20, color="orangered", alpha=0.7, edgecolor="white")
axes[1].axvline(jul_cdd.mean(), color="darkred", linestyle="--", label=f"Mean = {jul_cdd.mean():.1f}")
axes[1].set_title("July Daily CDD Distribution")
axes[1].set_xlabel("CDD")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"1月の日次HDD: 平均={jan_hdd.mean():.1f}, 標準偏差={jan_hdd.std():.1f}")
print(f"7月の日次CDD: 平均={jul_cdd.mean():.1f}, 標準偏差={jul_cdd.std():.1f}")
print(f"\n→ 1月のHDD先物 ≈ 日次HDD平均 × 31日 ≈ {jan_hdd.mean() * 31:.0f}")
print(f"→ 7月のCDD先物 ≈ 日次CDD平均 × 31日 ≈ {jul_cdd.mean() * 31:.0f}")

## 7. 次のステップ

このNotebookで学んだこと:
1. 気温データには強い**季節性**がある（株価にはない特徴）
2. **HDD/CDD**は気温リスクを定量化する指標
3. CME先物は**月次累積HDD/CDD**に対して取引される
4. 分布の形状が**プライシング手法の選択**に影響する

**Step 2 (03_ou_process.ipynb)** では:
- 季節成分を除去して「残差」を分析
- 残差がOrnstein-Uhlenbeck (OU) 過程に従うことを確認
- OU過程のパラメータ (κ, σ) を推定
- モンテカルロシミュレーションで気温パスを生成

**Step 3 (04_pricing.ipynb)** では:
- Burn Analysis（過去実績ベースのプライシング）
- Monte Carlo法によるHDD/CDD先物・オプションの価格計算